# 03. Weighted Preference State and Multi-Surface Click Paths

Earlier notebooks used preference sets. This notebook uses the richer
weighted profile-state fields:

- `interested_categories`
- `preferred_news_sources`

These fields are current profile snapshots. They are useful audit signals,
but they are not edit histories or causal records of preference change.


<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.

**Public-package behavior:** If authorized processed data are unavailable, this notebook skips private-data computation cells without raising an error. This silent skip is expected and does not reproduce manuscript estimates. For a fully runnable bundled-data example, open `../01_metric_walkthrough_synthetic.ipynb` or run `python scripts/run_public_synthetic_pipeline.py` from the repository root.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:weighted_pref_guide -->

## Reader guide: why weighted profile state matters

Earlier notebooks treat preferences as sets. This notebook uses the richer **weighted profile-state** fields. These fields are closer to a probability distribution over categories or sources, so they can be compared with clicked distributions using cosine similarity and preference mass.

Important limitation: the available profile table is a current snapshot, not a full edit history. Therefore weighted profile-state alignment is interpreted as an audit signal, not as proof that the profile state caused later consumption.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:surface_pathways_guide -->

## Surface pathways interpretation

Mobile news apps are not single ranked lists. Users can click through headline modules, category tabs, newsroom pages, home/feed views, search, notifications, and article-detail paths. Surface analysis asks whether different entry pathways show different traceability and preference alignment.

This matters because a click outside the logged recommendation list is not automatically a recommender failure. It may enter through another interface pathway that the recommendation log does not capture.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "03_weighted_preferences_and_surface_paths"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

PRIVATE_DATA_AVAILABLE = PROCESSED.exists()

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PRIVATE_DATA_AVAILABLE}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PRIVATE_DATA_AVAILABLE:
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`. Remaining private-data computation cells will be skipped; use the public synthetic pipeline for a fully runnable bundled-data example."
    ))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    def safe_parse(value):
        '''Parse JSON-like strings stored in CSV cells.''';
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return None
        text = str(value).strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                continue
        return None
    
    
    def truthy(series: pd.Series) -> pd.Series:
        '''Normalize common true-like CSV values to booleans.''';
        return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])
    
    
    def pref_set(row: pd.Series, primary: str, fallback: str) -> set[str]:
        '''Return a stated preference set from explicit selections or weighted profile state.
    
        primary is usually the user-selection column. fallback is the current weighted
        profile-state column. For a weighted dictionary, small positive weights are
        treated as selected dimensions for set-overlap analyses.
        '''
        parsed = safe_parse(row.get(primary, ""))
        if parsed is None:
            parsed = safe_parse(row.get(fallback, ""))
        if isinstance(parsed, dict):
            out = set()
            for key, value in parsed.items():
                try:
                    if float(value) > 0.01:
                        out.add(str(key))
                except Exception:
                    out.add(str(key))
            return out or set(map(str, parsed.keys()))
        if isinstance(parsed, list):
            out = set()
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    out.add(str(item["value"]))
                else:
                    out.add(str(item))
            return out
        return set()
    
    
    def normalize_dist(values: dict[str, float]) -> dict[str, float]:
        clean = {}
        for key, value in values.items():
            try:
                val = float(value)
            except Exception:
                continue
            if key and val > 0:
                clean[str(key)] = val
        total = sum(clean.values())
        if total <= 0:
            return {}
        return {key: val / total for key, val in clean.items()}
    
    
    def parse_weight_dist(value) -> dict[str, float]:
        '''Parse a weighted preference dictionary such as {'Politics': 0.7, ...}.''';
        parsed = safe_parse(value)
        if isinstance(parsed, dict):
            return normalize_dist({str(k): v for k, v in parsed.items()})
        if isinstance(parsed, list):
            vals = []
            for item in parsed:
                if isinstance(item, dict) and "value" in item:
                    vals.append(str(item["value"]))
                else:
                    vals.append(str(item))
            if vals:
                return {key: 1 / len(vals) for key in vals if key}
        return {}
    
    
    def norm_entropy(values) -> float:
        '''Normalized entropy in [0, 1]. Higher means more diverse.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        counts = values.value_counts().to_numpy(dtype=float)
        probs = counts / counts.sum()
        ent = -(probs * np.log(probs)).sum()
        max_ent = np.log(len(counts)) if len(counts) > 1 else 1.0
        return float(ent / max_ent) if max_ent else 0.0
    
    
    def norm_entropy_dist(dist: dict[str, float]) -> float:
        values = np.array([v for v in dist.values() if v > 0], dtype=float)
        if len(values) == 0:
            return 0.0
        values = values / values.sum()
        ent = float(-(values * np.log(values)).sum())
        denom = math.log(len(values)) if len(values) > 1 else 1.0
        return ent / denom if denom else 0.0
    
    
    def top_share(values) -> float:
        '''Share of the most common item. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        return float(values.value_counts().iloc[0] / len(values))
    
    
    def hhi(values) -> float:
        '''Herfindahl-Hirschman concentration index. Higher means more concentrated.''';
        values = pd.Series(values).dropna().astype(str)
        values = values[values != ""]
        if values.empty:
            return np.nan
        probs = values.value_counts(normalize=True).to_numpy(dtype=float)
        return float(np.sum(probs * probs))
    
    
    def jaccard(a: set[str], b: set[str]) -> float:
        if not a or not b:
            return np.nan
        return len(a & b) / len(a | b)
    
    
    def distribution_from_values(values) -> dict[str, float]:
        series = pd.Series(values).dropna().astype(str)
        series = series[series != ""]
        if series.empty:
            return {}
        counts = series.value_counts()
        return {str(k): float(v) / float(counts.sum()) for k, v in counts.items()}
    
    
    def cosine(a: dict[str, float], b: dict[str, float]) -> float:
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        av = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        bv = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        denom = np.linalg.norm(av) * np.linalg.norm(bv)
        return float(np.dot(av, bv) / denom) if denom > 0 else np.nan
    
    
    def js_divergence(a: dict[str, float], b: dict[str, float]) -> float:
        '''Jensen-Shannon divergence between two discrete distributions.''';
        keys = sorted(set(a) | set(b))
        if not keys:
            return np.nan
        p = np.array([a.get(k, 0.0) for k in keys], dtype=float)
        q = np.array([b.get(k, 0.0) for k in keys], dtype=float)
        if p.sum() <= 0 or q.sum() <= 0:
            return np.nan
        p = p / p.sum()
        q = q / q.sum()
        m = 0.5 * (p + q)
        def kl(x, y):
            mask = x > 0
            return float((x[mask] * np.log2(x[mask] / y[mask])).sum())
        return 0.5 * kl(p, m) + 0.5 * kl(q, m)
    
    
    def dist_top_key(dist: dict[str, float]) -> str:
        return max(dist.items(), key=lambda item: item[1])[0] if dist else ""
    
    
    def classify_surface(row: pd.Series) -> str:
        '''Map raw app parameters to the surface categories used in the paper.''';
        source = str(row.get("p_source", "") or "").lower()
        section = str(row.get("section", "") or "").lower()
        message = str(row.get("message_type", "") or "").lower()
        is_headline = str(row.get("is_headline", "") or "").lower() in {"1.0", "1", "true", "yes"}
        if "push" in message or "notification" in source or "notification" in section:
            return "notification"
        if is_headline or "headline" in section:
            return "headline"
        if section.startswith("category_"):
            return "category"
        if "for_you" in section or source == "home":
            return "home/feed"
        if "related" in section or source == "news_detail":
            return "article-detail"
        if "publisher" in section or source in {"newsroom", "source"}:
            return "newsroom"
        if "podcast" in section or "podcast" in source:
            return "podcast"
        if "insight" in section or "insight" in source:
            return "insights"
        if "search" in section or source == "search":
            return "search"
        return "other/unknown"
    
    
    def mask_id(value: str) -> str:
        '''Mask user identifiers before showing examples in a notebook.''';
        text = str(value)
        if len(text) <= 8:
            return text[:2] + "***"
        return text[:4] + "***" + text[-4:]


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    profiles = pd.read_csv(PROCESSED / "user_profiles_clean.csv", dtype=str).fillna("")
    clicks = pd.read_csv(PROCESSED / "click_events_enriched.csv", dtype=str).fillna("")
    matched_clicks = clicks.loc[truthy(clicks["news_matched"])].copy()
    profile_map = profiles.set_index("user_id")


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Example of how one weighted profile-state distribution is parsed.
    example = profiles.loc[profiles["interested_categories"].astype(str).str.contains("{", regex=False)].iloc[0]
    cat_pref = parse_weight_dist(example["interested_categories"])
    display(pd.DataFrame(sorted(cat_pref.items(), key=lambda x: x[1], reverse=True), columns=["category", "weight"]).head(10))


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Compute weighted preference-click alignment for each user.
    rows = []
    for user_id, group in matched_clicks.groupby("user_id"):
        if user_id not in profile_map.index:
            continue
        profile = profile_map.loc[user_id]
        cat_pref = parse_weight_dist(profile.get("interested_categories", ""))
        pub_pref = parse_weight_dist(profile.get("preferred_news_sources", ""))
        cat_click = distribution_from_values(group["category_final"])
        pub_click = distribution_from_values(group["publisher_id_final"])
        if not cat_click:
            continue
    
        top_click_cat = dist_top_key(cat_click)
        top_pref_cat = dist_top_key(cat_pref)
        top_click_pub = dist_top_key(pub_click)
        top_pref_pub = dist_top_key(pub_pref)
        top3_click_cats = [key for key, _ in Counter(group["category_final"].dropna().astype(str)).most_common(3)]
        top5_click_pubs = [key for key, _ in Counter(group["publisher_id_final"].dropna().astype(str)).most_common(5)]
    
        rows.append({
            "user_id": user_id,
            "masked_user": mask_id(user_id),
            "weighted_click_count": len(group),
            "category_pref_entropy_weighted": norm_entropy_dist(cat_pref),
            "category_pref_max_weight": max(cat_pref.values()) if cat_pref else 0.0,
            "category_pref_salient_count": sum(v > 0.01 for v in cat_pref.values()),
            "publisher_pref_entropy_weighted": norm_entropy_dist(pub_pref),
            "publisher_pref_max_weight": max(pub_pref.values()) if pub_pref else 0.0,
            "publisher_pref_salient_count": sum(v > 0.001 for v in pub_pref.values()),
            "top_clicked_category_weight": cat_pref.get(top_click_cat, 0.0),
            "top_preference_category_click_share": cat_click.get(top_pref_cat, 0.0),
            "top3_clicked_category_pref_mass": sum(cat_pref.get(k, 0.0) for k in top3_click_cats),
            "category_pref_click_cosine": cosine(cat_pref, cat_click),
            "category_pref_click_js": js_divergence(cat_pref, cat_click),
            "top_clicked_publisher_weight": pub_pref.get(top_click_pub, 0.0),
            "top_preference_publisher_click_share": pub_click.get(top_pref_pub, 0.0),
            "top5_clicked_publisher_pref_mass": sum(pub_pref.get(k, 0.0) for k in top5_click_pubs),
            "publisher_pref_click_cosine": cosine(pub_pref, pub_click),
            "publisher_pref_click_js": js_divergence(pub_pref, pub_click),
            "top_preference_category_equals_top_click": int(top_pref_cat == top_click_cat and bool(top_pref_cat)),
            "top_preference_publisher_equals_top_click": int(top_pref_pub == top_click_pub and bool(top_pref_pub)),
        })
    
    weighted = pd.DataFrame(rows)
    display(weighted.drop(columns=["user_id"]).head(10))
    display(weighted.describe().T)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    saved_weighted = pd.read_csv(TABLES / "preference_weighted_features.csv")
    compare = pd.DataFrame([
        {
            "metric": "n_users",
            "rebuilt": len(weighted),
            "saved": len(saved_weighted),
        },
        {
            "metric": "category_pref_click_cosine_mean",
            "rebuilt": weighted["category_pref_click_cosine"].mean(),
            "saved": saved_weighted["category_pref_click_cosine"].mean(),
        },
        {
            "metric": "top3_clicked_category_pref_mass_mean",
            "rebuilt": weighted["top3_clicked_category_pref_mass"].mean(),
            "saved": saved_weighted["top3_clicked_category_pref_mass"].mean(),
        },
    ])
    display(compare)


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Surface pathway analysis: where did clicks happen?
    matched_clicks["surface"] = matched_clicks.apply(classify_surface, axis=1)
    surface = matched_clicks.groupby("surface").agg(
        click_rows=("news_id", "size"),
        users=("user_id", "nunique"),
        median_position=("position", lambda s: pd.to_numeric(s, errors="coerce").median()),
    ).assign(click_share=lambda d: d["click_rows"] / d["click_rows"].sum()).sort_values("click_rows", ascending=False)
    
    display(surface)
    ax = surface["click_share"].head(8).sort_values().plot(kind="barh", figsize=(8, 4), color="#0F766E")
    ax.set_title("Observed click consumption by app surface")
    ax.set_xlabel("Share of matched clicks")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # Preference-click cosine by surface. This asks whether clicks on each
    # surface align with the current weighted category profile.
    rows = []
    for (user_id, surface_name), group in matched_clicks.groupby(["user_id", "surface"]):
        if len(group) < 2 or user_id not in profile_map.index:
            continue
        cat_pref = parse_weight_dist(profile_map.loc[user_id].get("interested_categories", ""))
        cat_click = distribution_from_values(group["category_final"])
        if not cat_pref or not cat_click:
            continue
        rows.append({
            "user_id": user_id,
            "surface": surface_name,
            "click_rows": len(group),
            "category_pref_click_cosine": cosine(cat_pref, cat_click),
            "top_clicked_category_weight": cat_pref.get(dist_top_key(cat_click), 0.0),
        })
    
    surface_align_user = pd.DataFrame(rows)
    surface_align = surface_align_user.groupby("surface").agg(
        user_surface_groups=("user_id", "nunique"),
        total_click_rows=("click_rows", "sum"),
        pref_click_cosine=("category_pref_click_cosine", "mean"),
        top_clicked_category_weight=("top_clicked_category_weight", "mean"),
    ).sort_values("total_click_rows", ascending=False)
    display(surface_align)


**Interpretive boundary**

Weighted profile-state features make the audit stronger because they are
richer than set membership. But they remain a current snapshot. They should
not be interpreted as a complete history of how preferences changed over
time.


## Step map: from set preferences to weighted profile-state and surfaces

Notebook 01 treated preference mostly as a set: category is either in the stated set or not. Notebook 03 uses the richer profile-state representation, where categories and sources have weights.

This matters because two users can both include `Politics`, but one may have 80% of profile mass on Politics while another has 15%. Weighted cosine uses that intensity information.

The surface part asks a different question: even if a clicked article is related to a user's profile, it may have been reached through headline, category, home/feed, newsroom, article-detail, search, notification, or unknown paths. That is why the paper separates surface pathways from the substantive P/E/C layers.

In [ ]:
if PRIVATE_DATA_AVAILABLE:
    notebook03_flow = pd.DataFrame([
        {
            "stage": "1. Parse weighted profile-state dictionaries",
            "input": "profiles.interested_categories and profiles.preferred_news_sources",
            "operation": "convert JSON-like weight dictionaries into category/source vectors",
            "output": "cat_pref, pub_pref per user",
            "paper meaning": "preference is not only membership; it also has intensity.",
        },
        {
            "stage": "2. Build click distributions",
            "input": "matched_clicks category_final and publisher_id_final",
            "operation": "count clicked categories/sources and normalize to distributions",
            "output": "cat_click, pub_click per user",
            "paper meaning": "actual consumption is represented as observed click distribution.",
        },
        {
            "stage": "3. Compute cosine similarity",
            "input": "preference vector + click distribution vector",
            "operation": "compare vector direction on the union of observed labels",
            "output": "category/source preference-click cosine",
            "paper meaning": "category profile has moderate directional signal; source signal is much weaker.",
        },
        {
            "stage": "4. Classify click surface",
            "input": "p_source, section, is_headline, position, message_type",
            "operation": "apply deterministic surface taxonomy rules",
            "output": "surface column and surface_summary",
            "paper meaning": "click consumption is multi-surface, not just one recommendation-list endpoint.",
        },
    ])
    display(notebook03_flow)


## What to look for in Notebook 03 outputs

- If category cosine is moderate but publisher/source cosine is weak, that does not mean the profile is useless. It means coarse category preference carries more signal than the much larger and sparser publisher/source label space.
- Surface summaries should be read as click pathway composition, not as total exposure. We know where a click event was logged, but we do not know every viewport impression the user saw before clicking.
- Weighted profile state is still a snapshot. It is not versioned preference history. That limitation is why Figure 1 places profile snapshot under observability boundaries.

In [ ]:
if PRIVATE_DATA_AVAILABLE:
    # AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
    # This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
    # It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
    _save_names = ['weighted', 'compare', 'surface', 'surface_align_user', 'surface_align', 'notebook03_flow']
    _saved_paths = []
    for _name in _save_names:
        if _name not in globals():
            continue
        _obj = globals()[_name]
        _path = SCRATCH_TABLES / f"{_name}.csv"
        try:
            if isinstance(_obj, pd.DataFrame):
                _obj.to_csv(_path, index=False)
            elif isinstance(_obj, pd.Series):
                _obj.to_frame().to_csv(_path)
            elif isinstance(_obj, list):
                pd.DataFrame(_obj).to_csv(_path, index=False)
            elif isinstance(_obj, dict):
                pd.DataFrame([_obj]).to_csv(_path, index=False)
            else:
                continue
            _saved_paths.append(_path)
        except Exception as exc:
            print(f"Could not save {_name}: {exc}")
    
    try:
        import matplotlib.pyplot as _plt
        for _idx, _num in enumerate(_plt.get_fignums(), 1):
            _fig = _plt.figure(_num)
            _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
            _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
            _saved_paths.append(_fig_path)
    except Exception as exc:
        print(f"Could not save open matplotlib figures: {exc}")
    
    print("Saved scratch artifacts:")
    for _path in _saved_paths:
        print(" -", _path)
